# RPD03 - Training and Evaluating an ACT Policy  (SO-101 - LeRobot - ROCm)

### Lab Description

This lab trains your first autonomous policy: **ACT (Action Chunking with Transformers)**, a task-specific imitation-learning model trained from scratch on the demonstrations you recorded in RPD02. You first see why a pretrained ACT model for a *different* robot cannot simply be dropped onto the SO-101 (a hard dimension mismatch), then train ACT on *your own* data with `lerobot-train` and deploy it on the follower arm with `lerobot-record --policy.path` -- watching it execute the task with no human in the loop.

## Lab Overview

| Step | Topic | Key Concepts |
|------|-------|-------------|
| 1 | What is ACT? | Action Chunking, Transformer, CVAE |
| 2 | Verify the Environment | PyTorch, ROCm, GPU availability |
| 3 | Pretrained ACT Models | Hugging Face Hub, model download, config inspection |
| 4 | Experiment: Deploy Without Finetuning | Architecture mismatch, dimension errors, why zero-shot fails |
| 5 | Train ACT on Your Dataset | `lerobot-train`, local dataset, training parameters |
| 6 | Evaluate the Trained Policy | `lerobot-record` with `--policy.path`, autonomous execution, video playback |
| 7 | Pretrained vs Finetuned | Zero-shot limitations, why task-specific training matters |

#### Recommended Hardware

- Two SO-101 arms (one leader, one follower) with Feetech STS3215 servos
- USB hub connecting both arms to the host machine
- Two USB cameras (front and side views)
- GPU with ROCm support (for training)

#### Software Environment

Docker container built from `Dockerfile` with [LeRobot](https://github.com/huggingface/lerobot) v0.5.0+ and ROCm PyTorch.\
Launch with `run.sh` (GPU + USB devices mounted).

#### Prerequisites

- Completed **RPD01** (teleoperation working, calibration verified)
- Completed **RPD02** (dataset recorded at `/opt/workspace/lerobot/local_data/my_dataset`)
- At least **50 episodes** recommended for reasonable training results

## Goals

1. **Understand ACT**: Learn what Action Chunking with Transformers is and why it works well for manipulation tasks.
2. **Explore Pretrained Models**: Download a pretrained ACT model from Hugging Face Hub and understand its limitations.
3. **Train a Policy**: Train an ACT policy from scratch on your demonstration dataset.
4. **Evaluate on the Robot**: Deploy the trained policy on the follower arm and observe autonomous behavior.
5. **Compare Results**: Understand why a pretrained model for a different robot cannot be used directly.

## 1. What is ACT?

**ACT (Action Chunking with Transformers)** is an imitation learning algorithm introduced by Tony Zhao et al. in *"Learning Fine-Grained Bimanual Manipulation with Low-Cost Hardware"* (RSS 2023). It is designed to learn dexterous manipulation policies from human demonstrations.

### Key Ideas

**Action Chunking** -- Instead of predicting one action per timestep, ACT predicts a **chunk** of future actions at once (e.g. the next 100 timesteps). This helps the policy:

- Avoid compounding errors from single-step predictions
- Produce smoother, more coherent motions
- Handle temporally correlated behaviors (e.g. reaching then grasping)

**Conditional VAE (CVAE)** -- During training, ACT uses a variational autoencoder to capture the multimodality in human demonstrations. Different people may perform the same task differently, and the CVAE encodes this variation into a latent style variable. At inference time, the style variable is set to zero (the prior mean), producing the average behavior.

**Transformer Architecture** -- ACT uses a Transformer encoder-decoder:

- The **encoder** processes current observations (joint positions + camera images) along with a latent style variable
- The **decoder** outputs a chunk of future joint positions (actions)
- Camera images are processed through a vision backbone (ResNet18 by default) before being fed to the Transformer

### Architecture Diagram

```
Camera Images ---> [ResNet18] ---> Image Tokens
                                       |
Joint Positions ---------------------->|
                                       v
Latent Style (z) -------> [Transformer Encoder] ---> [Transformer Decoder] ---> Action Chunk
                                                                                 (next K steps)
```

### Why ACT for SO-101?

ACT is well-suited for our setup because:

- It works with **low-cost hardware** (designed for similar 6-DOF arms)
- It learns from a **small number of demonstrations** (50-100 episodes)
- It handles **vision-based tasks** with multiple camera views
- It runs **real-time inference** on a GPU

## 2. Verify the Environment

In [15]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))

PyTorch version: 2.9.1+rocm7.11.0
CUDA available: True
GPU device: Radeon 8060S Graphics


## 3. Pretrained ACT Models

Hugging Face Hub hosts several pretrained ACT models. In this section, we download one to understand the model structure and explore why zero-shot transfer across different robots does not work.

### Download a Pretrained Model

The model `lerobot/act_aloha_sim_transfer_cube_human` was trained on the ALOHA simulation environment for a cube transfer task. We will download it to `/opt/workspace/lerobot/models/` to inspect its structure.

In [16]:
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path("/opt/workspace/lerobot/models/act_aloha_pretrained")

if not model_dir.exists():
    print("Downloading pretrained ACT model from Hugging Face Hub...")
    snapshot_download(
        repo_id="lerobot/act_aloha_sim_transfer_cube_human",
        local_dir=str(model_dir),
    )
    print(f"Model saved to: {model_dir}")
else:
    print(f"Model already exists at: {model_dir}")

print("\nModel files:")
for f in sorted(model_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.relative_to(model_dir)}  ({size_mb:.1f} MB)")

Model already exists at: /opt/workspace/lerobot/models/act_aloha_pretrained

Model files:
  .cache/huggingface/.gitignore  (0.0 MB)
  .cache/huggingface/download/.gitattributes.lock  (0.0 MB)
  .cache/huggingface/download/.gitattributes.metadata  (0.0 MB)
  .cache/huggingface/download/README.md.lock  (0.0 MB)
  .cache/huggingface/download/README.md.metadata  (0.0 MB)
  .cache/huggingface/download/config.json.lock  (0.0 MB)
  .cache/huggingface/download/config.json.metadata  (0.0 MB)
  .cache/huggingface/download/demo.gif.lock  (0.0 MB)
  .cache/huggingface/download/demo.gif.metadata  (0.0 MB)
  .cache/huggingface/download/eval_info.json.lock  (0.0 MB)
  .cache/huggingface/download/eval_info.json.metadata  (0.0 MB)
  .cache/huggingface/download/model.safetensors.lock  (0.0 MB)
  .cache/huggingface/download/model.safetensors.metadata  (0.0 MB)
  .cache/huggingface/download/train_config.json.lock  (0.0 MB)
  .cache/huggingface/download/train_config.json.metadata  (0.0 MB)
  .cache/hugging

In [17]:
import json

config_path = Path("/opt/workspace/lerobot/models/act_aloha_pretrained/config.json")

if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)

    print("=== Pretrained Model Configuration ===\n")
    print(f"  Policy type:     {config.get('type', 'N/A')}")
    print(f"  Chunk size:      {config.get('chunk_size', 'N/A')}")
    print(f"  Input shapes:    {json.dumps(config.get('input_shapes', {}), indent=18)}")
    print(f"  Output shapes:   {json.dumps(config.get('output_shapes', {}), indent=18)}")
else:
    print("config.json not found. Run the download cell above first.")

=== Pretrained Model Configuration ===

  Policy type:     act
  Chunk size:      100
  Input shapes:    {}
  Output shapes:   {}


## 4. Experiment: Deploying Without Finetuning

Before training our own model, let's see what happens if we try to use the pretrained ALOHA model directly on our SO-101 robot. This experiment demonstrates why finetuning (or training from scratch) is necessary.

### Compare Model vs Robot Specifications

The following cell loads the pretrained model's config and compares it with our dataset's structure:

In [18]:
import json
from pathlib import Path

pretrained_dir = Path("/opt/workspace/lerobot/models/act_aloha_pretrained")
dataset_dir = Path("/opt/workspace/lerobot/local_data/my_dataset")

with open(pretrained_dir / "config.json") as f:
    pretrained_cfg = json.load(f)

with open(dataset_dir / "meta" / "info.json") as f:
    dataset_info = json.load(f)

input_feats = pretrained_cfg.get("input_features", {})
output_feats = pretrained_cfg.get("output_features", {})

print("=" * 60)
print("PRETRAINED MODEL (ALOHA)")
print("=" * 60)
for key, val in input_feats.items():
    print(f"  Input:  {key:40s} -> shape={val['shape']}, type={val['type']}")
for key, val in output_feats.items():
    print(f"  Output: {key:40s} -> shape={val['shape']}, type={val['type']}")

print()
print("=" * 60)
print("OUR DATASET (SO-101)")
print("=" * 60)
features = dataset_info.get("features", {})
for key, val in features.items():
    if "shape" in val:
        print(f"  {key:45s} -> shape={val['shape']}, dtype={val.get('dtype', 'N/A')}")

print()
print("=" * 60)
print("MISMATCH SUMMARY")
print("=" * 60)

pretrained_action_dim = output_feats.get("action", {}).get("shape", [None])[0]
our_action_dim = features.get("action", {}).get("shape", [None])[0]

pretrained_state_dim = input_feats.get("observation.state", {}).get("shape", [None])[0]
our_state_dim = features.get("observation.state", {}).get("shape", [None])[0]

pretrained_cameras = [k for k in input_feats if "image" in k.lower()]
our_cameras = [k for k in features if "image" in k.lower()]

print(f"  Action dims:  pretrained={pretrained_action_dim}  vs  ours={our_action_dim}")
print(f"  State dims:   pretrained={pretrained_state_dim}  vs  ours={our_state_dim}")
print(f"  Camera keys:  pretrained={pretrained_cameras}")
print(f"                ours={our_cameras}")

mismatches = []
if pretrained_action_dim != our_action_dim:
    mismatches.append(f"Action: {pretrained_action_dim} vs {our_action_dim}")
if pretrained_state_dim != our_state_dim:
    mismatches.append(f"State: {pretrained_state_dim} vs {our_state_dim}")
if set(pretrained_cameras) != set(our_cameras):
    mismatches.append(f"Cameras: {pretrained_cameras} vs {our_cameras}")

if mismatches:
    print()
    print("  >> INCOMPATIBLE -- model CANNOT be used on this robot:")
    for m in mismatches:
        print(f"     - {m}")

PRETRAINED MODEL (ALOHA)
  Input:  observation.images.top                   -> shape=[3, 480, 640], type=VISUAL
  Input:  observation.state                        -> shape=[14], type=STATE
  Output: action                                   -> shape=[14], type=ACTION

OUR DATASET (SO-101)
  action                                        -> shape=[6], dtype=float32
  observation.state                             -> shape=[6], dtype=float32
  observation.images.front                      -> shape=[480, 640, 3], dtype=video
  observation.images.side                       -> shape=[480, 640, 3], dtype=video
  timestamp                                     -> shape=[1], dtype=float32
  frame_index                                   -> shape=[1], dtype=int64
  episode_index                                 -> shape=[1], dtype=int64
  index                                         -> shape=[1], dtype=int64
  task_index                                    -> shape=[1], dtype=int64

MISMATCH SUMMARY
 

### What Happens If You Try Anyway?

If you attempt to deploy the pretrained ALOHA model on the SO-101 robot:

```bash
lerobot-record \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm \
    --robot.calibration_dir=/opt/workspace/lerobot/calibration \
    --robot.cameras='{ front: {type: opencv, index_or_path: /dev/video3, width: 640, height: 480, fps: 30, fourcc: MJPG, warmup_s: 3}, side: {type: opencv, index_or_path: /dev/video1, width: 640, height: 480, fps: 30, fourcc: MJPG, warmup_s: 3}}' \
    --display_data=false \
    --dataset.repo_id=local/eval_pretrained \
    --dataset.root=/opt/workspace/lerobot/local_data/eval_pretrained \
    --dataset.num_episodes=1 \
    --dataset.single_task="Pick up the object" \
    --dataset.push_to_hub=false \
    --policy.path=/opt/workspace/lerobot/models/act_aloha_pretrained \
    --policy.device=cuda \
    --play_sounds=false
```

**This command will fail** with the following error:

```
RuntimeError: Error(s) in loading state_dict for ACTPolicy:
    size mismatch for model.vae_encoder_action_input_proj.weight:
        copying a param with shape torch.Size([512, 14]) from checkpoint,
        the shape in current model is torch.Size([512, 6]).
    size mismatch for model.action_head.weight:
        copying a param with shape torch.Size([14, 512]) from checkpoint,
        the shape in current model is torch.Size([6, 512]).
    size mismatch for model.action_head.bias:
        copying a param with shape torch.Size([14]) from checkpoint,
        the shape in current model is torch.Size([6]).
```

LeRobot reads our SO-101 dataset metadata (6 joints) and builds a 6-dimensional ACT model, then tries to load the pretrained ALOHA weights (14 joints) into it. The dimensions do not match, so loading fails immediately.

### Why This Matters

| Scenario | Result |
|:---|:---|
| Pretrained model on **same** robot + **same** task | Works well (best case) |
| Pretrained model on **same** robot + **different** task | May partially work, finetuning recommended |
| Pretrained model on **different** robot | **Fails completely** -- architecture mismatch |

> **Key takeaway:** ACT models are tightly coupled to the robot's hardware configuration (number of joints, camera setup). There is no shortcut -- you must **train on your own demonstrations** from your specific robot. This is exactly what we will do in the next section.

## 5. Train ACT on Your Dataset

Now we will train an ACT policy from scratch using the demonstration data collected in RPD02.

### Verify Dataset

First, confirm the dataset from RPD02 is available.

In [19]:
import json
from pathlib import Path

dataset_dir = Path("/opt/workspace/lerobot/local_data/my_dataset")

if dataset_dir.exists():
    info_path = dataset_dir / "meta" / "info.json"
    if info_path.exists():
        with open(info_path) as f:
            info = json.load(f)
        print(f"Dataset:         {dataset_dir}")
        print(f"Total episodes:  {info.get('total_episodes', 'N/A')}")
        print(f"Total frames:    {info.get('total_frames', 'N/A')}")
        print(f"FPS:             {info.get('fps', 'N/A')}")
        print(f"Features:        {list(info.get('features', {}).keys())}")
    else:
        print(f"Dataset directory exists but meta/info.json not found.")
else:
    print(f"Dataset not found at {dataset_dir}")
    print("Please complete Lab 2 first to record a dataset.")

Dataset:         /opt/workspace/lerobot/local_data/my_dataset
Total episodes:  20
Total frames:    17917
FPS:             30
Features:        ['action', 'observation.state', 'observation.images.front', 'observation.images.side', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']


### Run Training

Training must be run in a **JupyterLab Terminal** (File -> New -> Terminal) because it is a long-running process with continuous log output.

```bash
lerobot-train \
    --dataset.repo_id=local/my_dataset \
    --dataset.root=/opt/workspace/lerobot/local_data/my_dataset \
    --dataset.video_backend=pyav \
    --policy.type=act \
    --output_dir=/opt/workspace/lerobot/models/act_my_dataset \
    --job_name=act_my_dataset \
    --policy.device=cuda \
    --policy.push_to_hub=false \
    --wandb.enable=false
```

### Training Parameters

| Parameter | Description | Value |
|:---|:---|:---|
| `--dataset.repo_id` | Dataset identifier (logical name) | `local/my_dataset` |
| `--dataset.root` | Path to the local dataset from RPD02 | `/opt/workspace/lerobot/local_data/my_dataset` |
| `--dataset.video_backend` | Video decoder backend | `pyav` (default `torchcodec` has ABI issues with ROCm PyTorch) |
| `--policy.type` | Policy architecture | `act` |
| `--output_dir` | Where to save checkpoints and logs | `/opt/workspace/lerobot/models/act_my_dataset` |
| `--policy.device` | Device for training | `cuda` |
| `--policy.push_to_hub` | Upload trained model to Hub | `false` (set `true` with `--policy.repo_id` to push) |
| `--wandb.enable` | Enable Weights & Biases logging | `false` (set `true` if you have a W&B account) |

### ACT-Specific Parameters (Optional Overrides)

You can customize ACT's architecture and training by adding flags:

| Parameter | Default | Description |
|:---|:---|:---|
| `--policy.chunk_size` | 100 | Number of future actions predicted per step |
| `--policy.n_action_steps` | 100 | How many predicted actions are actually executed |
| `--policy.dim_model` | 512 | Transformer hidden dimension |
| `--policy.n_heads` | 8 | Number of attention heads |
| `--policy.use_vae` | true | Enable the CVAE latent variable |
| `--policy.kl_weight` | 10.0 | KL divergence loss weight |

> **Tip:** For a quick test run, reduce training steps with `--steps=1000`. For a proper policy, train for at least 50,000-100,000 steps.

### Monitoring Training Progress

Training logs appear in the terminal. Key metrics to watch:

```
step: 1000  loss: 0.0452  grad_norm: 12.3  lr: 1e-05  ...
step: 2000  loss: 0.0321  grad_norm: 8.7   lr: 1e-05  ...
```

- **loss** -- Should decrease over time. Lower loss means the policy better fits the demonstrations.
- **grad_norm** -- Gradient magnitude. Spikes may indicate instability.
- **lr** -- Learning rate.

Checkpoints are saved periodically to `--output_dir`. The final model is at:

```
/opt/workspace/lerobot/models/act_my_dataset/checkpoints/last/pretrained_model/
```

### Resuming Interrupted Training

If training is interrupted, resume from the last checkpoint:

```bash
lerobot-train \
    --dataset.repo_id=local/my_dataset \
    --dataset.root=/opt/workspace/lerobot/local_data/my_dataset \
    --dataset.video_backend=pyav \
    --policy.type=act \
    --output_dir=/opt/workspace/lerobot/models/act_my_dataset \
    --job_name=act_my_dataset \
    --policy.device=cuda \
    --policy.push_to_hub=false \
    --wandb.enable=false \
    --resume=true
```

### Verify Training Output

After training completes, verify that model files were saved.

In [20]:
from pathlib import Path

model_dir = Path("/opt/workspace/lerobot/models/act_my_dataset")
last_ckpt = model_dir / "checkpoints" / "last" / "pretrained_model"

if last_ckpt.exists():
    print(f"Trained model found at: {last_ckpt}\n")
    for f in sorted(last_ckpt.rglob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  {f.relative_to(last_ckpt)}  ({size_mb:.1f} MB)")
else:
    print(f"Model not found at {last_ckpt}")
    print("Make sure training has completed in Step 4.")

    ckpt_dir = model_dir / "checkpoints"
    if ckpt_dir.exists():
        ckpts = sorted(ckpt_dir.iterdir())
        if ckpts:
            print(f"\nAvailable checkpoints: {[c.name for c in ckpts]}")

Trained model found at: /opt/workspace/lerobot/models/act_my_dataset/checkpoints/last/pretrained_model

  config.json  (0.0 MB)
  model.safetensors  (197.1 MB)
  policy_postprocessor.json  (0.0 MB)
  policy_postprocessor_step_0_unnormalizer_processor.safetensors  (0.0 MB)
  policy_preprocessor.json  (0.0 MB)
  policy_preprocessor_step_3_normalizer_processor.safetensors  (0.0 MB)
  train_config.json  (0.0 MB)


## 6. Evaluate the Trained Policy

With the trained model, we can now deploy it on the real robot. The follower arm will execute actions **autonomously** based on the policy's predictions -- no leader arm or human teleoperation is needed.

We use `lerobot-record` with `--policy.path` to run the policy while recording evaluation episodes.

### Run Evaluation

Run the following in a **JupyterLab Terminal**:

```bash
rm -rf /opt/workspace/lerobot/local_data/eval_act

lerobot-record \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm \
    --robot.calibration_dir=/opt/workspace/lerobot/calibration \
    --robot.cameras='{ front: {type: opencv, index_or_path: /dev/video3, width: 640, height: 480, fps: 30, fourcc: MJPG, warmup_s: 3}, side: {type: opencv, index_or_path: /dev/video1, width: 640, height: 480, fps: 30, fourcc: MJPG, warmup_s: 3}}' \
    --display_data=false \
    --dataset.repo_id=local/eval_act \
    --dataset.root=/opt/workspace/lerobot/local_data/eval_act \
    --dataset.num_episodes=1 \
    --dataset.single_task="Pick up the object" \
    --dataset.episode_time_s=10 \
    --dataset.reset_time_s=1 \
    --dataset.push_to_hub=false \
    --dataset.vcodec=h264 \
    --dataset.streaming_encoding=true \
    --dataset.encoder_threads=2 \
    --policy.path=/opt/workspace/lerobot/models/act_my_dataset/checkpoints/last/pretrained_model \
    --policy.device=cuda \
    --play_sounds=false \
    2>&1 | grep -v "Corrupt JPEG" | grep -v "skipping action generation" | grep -v "running slower"
```

> **Important:** No `--teleop.*` parameters are needed. When `--policy.path` is set, the policy controls the robot autonomously. The leader arm is **not used** during evaluation.

> **Important:** Camera key names (`front`, `side`) must match **exactly** what was used during data collection in RPD02. Mismatched names will cause errors.

### Key Parameters

| Parameter | Value | Why |
|:---|:---|:---|
| `--dataset.num_episodes=1` | 1 episode per run | The robot keeps torque on during reset, so the gripper won't release between episodes. Run one at a time, then manually reset the scene. |
| `--dataset.episode_time_s=10` | 10 seconds | Should match the typical duration of your training demonstrations. The robot will hold still after completing the learned trajectory. |
| `--dataset.reset_time_s=1` | 1 second | Minimal reset time since we run one episode per run. |
| `--dataset.vcodec=h264` | H.264 codec | Much faster encoding than the default `libsvtav1` (AV1). Prevents the process from hanging between episodes. |
| `--dataset.streaming_encoding=true` | Encode in parallel | Video frames are encoded during recording instead of all at once after the episode, avoiding post-episode delays. |

### What to Expect

- The robot will **move on its own** based on the policy's predictions
- Each episode runs for `episode_time_s` seconds, then stops
- The robot may **freeze after completing the grasping motion** -- this is normal. The policy has finished executing its learned trajectory.
- If the gripper locks and won't release: press **Ctrl+C** first, then unplug the follower's USB cable to cut power and release the gripper
- Evaluation episodes are recorded to `/opt/workspace/lerobot/local_data/eval_act/`
- Watch the robot carefully -- keep the emergency stop nearby

### Running Additional Episodes

To add more evaluation episodes without overwriting existing ones:

```bash
lerobot-record \
    ... (same parameters as above, but remove the rm -rf line) ...
    --resume=true \
    2>&1 | grep -v "Corrupt JPEG" | grep -v "skipping action generation" | grep -v "running slower"
```

### Using a Specific Checkpoint

If you want to evaluate an intermediate checkpoint instead of the final one:

```bash
--policy.path=/opt/workspace/lerobot/models/act_my_dataset/checkpoints/005000/pretrained_model
```

Replace `005000` with the checkpoint step number you want to evaluate.

### Visualize Evaluation Results

After evaluation completes, the recorded episodes (including camera videos) are saved in `/opt/workspace/lerobot/local_data/eval_act/`. The cell below finds and plays the evaluation videos directly in this notebook, so you can review the robot's autonomous behavior without leaving JupyterLab.

In [22]:
from pathlib import Path
from IPython.display import display, Video, HTML

eval_dir = Path("/opt/workspace/lerobot/local_data/eval_act")
video_dir = eval_dir / "videos"

if not video_dir.exists():
    print(f"No videos found at {video_dir}")
    print("Run the evaluation command above first.")
else:
    # Videos are stored in subdirectories: videos/observation.images.front/, videos/observation.images.side/
    videos = sorted(video_dir.rglob("*.mp4"))
    if not videos:
        print(f"No .mp4 files found under {video_dir}")
        print("Subdirectories found:")
        for d in sorted(video_dir.iterdir()):
            print(f"  {d}")
    else:
        print(f"Found {len(videos)} evaluation video(s):\n")
        for v in videos:
            size_mb = v.stat().st_size / (1024 * 1024)
            print(f"  {v.relative_to(video_dir)}  ({size_mb:.1f} MB)")

        for v in videos:
            label = v.relative_to(video_dir)
            display(HTML(f"<h4>{label}</h4>"))
            display(Video(str(v), embed=True, html_attributes="controls loop width=640"))

Found 2 evaluation video(s):

  observation.images.front/chunk-000/file-000.mp4  (0.6 MB)
  observation.images.side/chunk-000/file-000.mp4  (1.2 MB)


## 7. Pretrained vs Finetuned: Why Training on Your Own Data Matters

### The Zero-Shot Transfer Problem

A natural question is: *"Can I just download a pretrained ACT model and use it directly on my robot?"*

The answer is **no** for most cases, and here is why:

| Factor | Impact |
|:---|:---|
| **Robot morphology** | Different robots have different numbers of joints, joint ranges, and kinematics. A model trained on a 14-joint bimanual ALOHA robot produces 14-dimensional actions -- incompatible with our 6-joint SO-101. |
| **Camera configuration** | Models expect specific camera names and viewpoints. A model trained with a `top` camera will look for `observation.images.top`, while our setup provides `observation.images.front` and `observation.images.side`. |
| **Task specifics** | Even between identical robots, a model trained to transfer cubes will not know how to pick up other objects or operate in a different workspace layout. |
| **Observation statistics** | Joint position ranges and image distributions differ between setups. The model's input normalization will be mismatched. |

### What Works: Training on Your Own Demonstrations

By collecting demonstrations on your specific robot (RPD02) and training from scratch (Step 4 above), the model learns:

- Your robot's exact joint dimensions and ranges
- Your camera viewpoints and naming conventions
- Your specific task and workspace layout
- The correct input/output normalization statistics

### Improving Policy Performance

If the trained policy does not perform well, consider:

| Strategy | Details |
|:---|:---|
| **Collect more data** | 50 episodes is the minimum. 100-200 episodes typically yield much better results. |
| **Improve data quality** | Ensure demonstrations are consistent -- same starting pose, same grasping strategy, objects in similar positions. |
| **Adjust camera angles** | Make sure the target object is clearly visible in both camera views across all episodes. |
| **Train longer** | Increase training steps (e.g. 100,000-200,000). Monitor the loss curve for convergence. |
| **Tune chunk_size** | Smaller chunks (e.g. 50) may work better for short, quick tasks. Larger chunks (100) help with longer motions. |
| **Check normalization** | Ensure the dataset stats (`meta/stats.json`) are reasonable -- extreme values may indicate recording issues. |

### Finetuning from a Compatible Pretrained Model

If a pretrained ACT model exists for the **same robot type and camera configuration** (e.g. another SO-101 model trained on a different task), you can finetune it:

```bash
lerobot-train \
    --policy.path=/opt/workspace/lerobot/models/existing_compatible_model \
    --dataset.repo_id=local/my_dataset \
    --dataset.root=/opt/workspace/lerobot/local_data/my_dataset \
    --output_dir=/opt/workspace/lerobot/models/act_finetuned \
    --policy.device=cuda \
    --wandb.enable=false
```

> **Note:** When using `--policy.path`, do **not** also pass `--policy.type`. The policy type is inferred from the loaded model's `config.json`.

Finetuning from a compatible pretrained model can converge faster and sometimes yield better results, especially with limited data. However, the model and dataset must share the same observation and action structure.

### Conclusion

In this lab, you have:

- Learned what **ACT (Action Chunking with Transformers)** is and how it works
- Downloaded a pretrained ACT model from Hugging Face Hub and examined its structure
- Understood why **pretrained models cannot be directly transferred** to robots with different configurations
- **Trained an ACT policy from scratch** on your own demonstration dataset
- **Evaluated the trained policy** on the real robot in autonomous mode
- Explored strategies for **improving policy performance** and **finetuning from compatible models**

The key insight is that imitation learning policies like ACT are tightly coupled to the robot hardware and task. Collecting high-quality demonstrations on your specific setup is the most important step for achieving good autonomous performance.

In **RPD04**, you will finetune **SmolVLA** -- a Vision-Language-Action foundation model -- on the same dataset and compare its performance with ACT.

## Acknowledgements

This lab series builds on the [LeRobot](https://github.com/huggingface/lerobot) project -- the `lerobot-teleoperate`, `lerobot-record`, and `lerobot-train` tools, the **ACT** and **SmolVLA** policies -- and the open-hardware **SO-101** arm from [TheRobotStudio / Hugging Face](https://github.com/TheRobotStudio/SO-ARM100). Training and inference run on AMD Ryzen AI (Radeon gfx1152) with ROCm.


---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
